# ML07 · 第一個神經網路 MLP

用 PyTorch 親手搭一個多層感知器（multilayer perceptron, MLP），並跑完一個完整的訓練迴圈，從零建立「網路如何學習」的直覺。

## 學習目標
- 理解多層感知器的基本結構：輸入層、隱藏層（hidden layer）、輸出層，以及為何需要非線性激活（nonlinear activation）。
- 學會用子類化（subclassing）nn.Module（__init__ / forward）定義自己的模型，也會用 nn.Sequential 快速堆疊。
- 認識常用積木：nn.Linear、ReLU，理解線性層與激活的分工。
- 搞懂損失函式（loss function）與優化器（optimizer）各自負責什麼。
- 能獨立寫出標準訓練迴圈：forward -> loss -> backward -> optimizer.step -> zero_grad，並說明每一步的意義。
- 用自造資料完成一個小迴歸或分類任務，觀察損失下降代表「正在學」。

In [ ]:
# 概念：載入本書共用工具，並固定亂數種子讓結果可重現
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)              # 固定 PyTorch 亂數，權重初始化每次相同，方便對照
np.random.seed(0)                 # 固定 numpy 亂數，自造資料每次相同

# 技巧：matplotlib 預設字型不含中文，圖上一律用英文標籤，避免出現方塊亂碼
plt.rcParams["axes.unicode_minus"] = False

print("torch 版本:", torch.__version__)
print("numpy 版本:", np.__version__)

## 從感知器到多層感知器

感知器（perceptron）是最簡單的神經元：把輸入加權求和再判斷，本質上只能畫出一條直線當分界。多層感知器（MLP）在中間多疊了隱藏層與非線性激活，才能逼近彎曲的關係。

關鍵直覺：線性層接線性層，數學上仍等於一個線性層（線性 + 線性還是線性）。唯有在中間插入非線性激活，網路才真正有表達力（expressive power），能畫出彎曲的邊界。

為什麼先學這個：理解「為何需要隱藏層與激活」，後面所有結構選擇才有依據。

In [ ]:
# 概念：自造一組無法用單一直線分開的點，對比直線邊界與彎曲邊界的差異
rng = np.random.default_rng(0)

# 造兩個交錯的區塊：A 類分布在左下與右上，B 類分布在左上與右下（典型的 XOR 形狀）
n_per_corner = 40
centers_a = [(-2.0, -2.0), (2.0, 2.0)]    # A 類兩個角落
centers_b = [(-2.0, 2.0), (2.0, -2.0)]    # B 類兩個角落

def make_blob(center, n):
    cx, cy = center
    return np.column_stack([rng.normal(cx, 0.7, n), rng.normal(cy, 0.7, n)])

points_a = np.vstack([make_blob(c, n_per_corner) for c in centers_a])
points_b = np.vstack([make_blob(c, n_per_corner) for c in centers_b])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax in axes:
    ax.scatter(points_a[:, 0], points_a[:, 1], c="tab:blue", label="class A", s=18)
    ax.scatter(points_b[:, 0], points_b[:, 1], c="tab:orange", label="class B", s=18)

# 左圖：一條直線（單層感知器能做到的極限），怎麼擺都分不開交錯的兩類
xs = np.linspace(-4, 4, 50)
axes[0].plot(xs, -xs, "k--", label="one straight line")   # 直線分界示意
axes[0].set_title("single line cannot separate")

# 右圖：一條彎曲邊界（MLP 能學到的），才能把交錯的兩類分開
theta = np.linspace(0, 2 * np.pi, 200)
axes[1].plot(2.6 * np.cos(theta), 2.6 * np.sin(theta), "k--", label="curved boundary")
axes[1].set_title("curved boundary can separate")

for ax in axes:
    ax.legend(loc="upper right", fontsize=8)
    ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 用 nn.Module 定義模型

PyTorch 的模型都繼承自 nn.Module。它把模型拆成兩個職責：
- __init__（建構子）：宣告「這個模型有哪些層」，把層存成屬性。
- forward（前向傳播）：描述「一筆資料怎麼依序流過這些層」。

理解這個分工是看懂所有 PyTorch 模型的鑰匙：__init__ 準備積木，forward 規定資料流向。把層宣告成屬性後，PyTorch 會自動把它們的權重登記為模型參數（parameters），訓練時才找得到要更新的對象。

形狀：
```
class MyModel(nn.Module):
    def __init__(self): ...   # 準備有哪些層
    def forward(self, x): ... # 資料怎麼流過這些層
```

In [ ]:
# 概念：子類化 nn.Module 寫一個最小 MLP（輸入 -> 隱藏 -> 輸出），先不訓練只看 forward 輸出形狀
class SmallMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()                       # 注意：一定要先呼叫父類別建構子，否則層不會被登記
        self.hidden = nn.Linear(in_dim, hidden_dim)   # 輸入層到隱藏層的全連接
        self.act = nn.ReLU()                          # 非線性激活，賦予網路表達力
        self.output = nn.Linear(hidden_dim, out_dim)  # 隱藏層到輸出層

    def forward(self, x):
        x = self.hidden(x)     # 資料先過第一層線性
        x = self.act(x)        # 再過 ReLU 折一下
        x = self.output(x)     # 最後過輸出層
        return x

model = SmallMLP(in_dim=2, hidden_dim=8, out_dim=1)
print(model)               # 印出模型結構，看得到三個子模組

# 模型參數數量：把每個參數張量的元素個數加總
n_params = sum(p.numel() for p in model.parameters())
print("參數總數:", n_params)

# 造 5 筆 2 維假輸入，跑一次 forward 看輸出形狀（此時權重是隨機的，數值無意義）
dummy = torch.randn(5, 2)
out = model(dummy)         # 等同 model.forward(dummy)，但慣例用 model(x)
print("輸入 shape:", dummy.shape, " 輸出 shape:", out.shape)

## 網路積木：Linear、ReLU、Sequential

把模型拆成三種積木更好理解：
- nn.Linear：全連接層（fully connected layer），做加權組合，內含權重（weight）與偏置（bias）。
- ReLU：整流線性單元（rectified linear unit），把負值壓成 0、正值保留，提供非線性「開關」。
- nn.Sequential：序列容器，把積木依宣告順序自動串起來，省去手寫 forward。

關鍵是維度要接得起來：前一層的輸出維度，必須等於後一層的輸入維度。下面用 Sequential 重寫上一節的同一個 MLP，確認兩種寫法等價。

In [ ]:
# 概念：用 nn.Sequential 把 Linear-ReLU-Linear 串成 MLP，並與子類化版本對照確認等價
seq_model = nn.Sequential(
    nn.Linear(2, 8),    # 輸入 2 維 -> 隱藏 8 維
    nn.ReLU(),          # 非線性
    nn.Linear(8, 1),    # 隱藏 8 維 -> 輸出 1 維（維度要與前一層的輸出對齊）
)
print(seq_model)

# 兩種寫法的參數數量應一致（結構相同）
seq_params = sum(p.numel() for p in seq_model.parameters())
print("Sequential 參數總數:", seq_params, " 子類化版本:", n_params)

# 概念：拆開看 Linear 內部的 weight 與 bias 形狀
first_linear = seq_model[0]                       # Sequential 可用索引取出某一層
print("第一層 weight shape:", first_linear.weight.shape)   # (輸出維度, 輸入維度)
print("第一層 bias shape:", first_linear.bias.shape)       # (輸出維度,)

# 跑一次 forward，確認輸出形狀與子類化版本相同
print("Sequential 輸出 shape:", seq_model(dummy).shape)

## 損失函式與優化器

訓練需要兩個角色分工：
- 損失函式（loss function）：給出「現在錯多少」的分數。迴歸常用均方誤差（mean squared error, MSE），分類常用交叉熵（cross-entropy）。
- 優化器（optimizer）：依梯度（gradient）決定「往哪個方向、走多大一步」去降低損失，也就是梯度下降（gradient descent）。學習率（learning rate）控制步伐大小。

交叉熵的直覺：它對「很有自信卻答錯」懲罰特別重，逼模型不要亂篤定。下面對同一筆假預測，分別算一次 MSE 與交叉熵，並建立一個 SGD 優化器看它綁定了哪些參數。

In [ ]:
# 概念：對同一筆假預測分別算 MSE 與交叉熵，並建立 SGD 觀察它綁定的參數
import torch.optim as optim

# 迴歸情境：自造 4 筆預測值與真實值（例如預測樓高）
pred_reg = torch.tensor([10.0, 22.0, 8.0, 31.0])
true_reg = torch.tensor([12.0, 20.0, 9.0, 30.0])
mse_loss = nn.MSELoss()
print("MSE:", mse_loss(pred_reg, true_reg).item())   # 平均每筆誤差的平方

# 分類情境：自造 3 筆、各 2 類的原始分數 logits，與正確類別索引
logits = torch.tensor([[2.0, 0.5],     # 第 1 筆比較傾向類別 0
                       [0.1, 1.8],     # 第 2 筆比較傾向類別 1
                       [1.0, 1.0]])    # 第 3 筆兩類分數相同（不確定）
targets = torch.tensor([0, 1, 0])      # 正確答案
ce_loss = nn.CrossEntropyLoss()        # 注意：輸入是未經 softmax 的原始 logits，不要自己先做 softmax
print("CrossEntropy:", ce_loss(logits, targets).item())

# 概念：優化器要被告知「該更新哪些參數」與「學習率」
optimizer = optim.SGD(seq_model.parameters(), lr=0.1)   # 把模型所有參數交給 SGD 管理
print("優化器管理的參數張量數量:", len(optimizer.param_groups[0]["params"]))
print("學習率 lr:", optimizer.param_groups[0]["lr"])

## 完整訓練迴圈

所有 PyTorch 訓練都是同一套五步驟骨架，每跑完整批資料一次叫一個訓練週期（epoch）：
1. forward：把資料餵進模型得到預測。
2. loss：用損失函式算出錯多少。
3. backward：反向傳播（backpropagation）算出每個參數的梯度。
4. optimizer.step：依梯度更新參數，往損失低處走一步。
5. zero_grad：把梯度歸零，準備下一輪。

特別注意：梯度預設會累加（accumulate）。若忘了 zero_grad，這一輪的梯度會疊到上一輪上，更新就會亂掉。下面對自造資料跑數十個 epoch，記錄並畫出損失下降曲線。

In [ ]:
# 概念：用五步驟骨架訓練一個迴歸 MLP，記錄每個 epoch 的損失並畫出下降曲線
# 自造資料：樓層數對總樓高，大致成正比再加雜訊（單一特徵的迴歸）
floors = np.linspace(3, 30, 60).reshape(-1, 1)              # 60 棟，樓層 3 到 30
heights = 3.2 * floors + 2.0 + np.random.normal(0, 2.0, floors.shape)   # 每層約 3.2 公尺加底座與雜訊

X = torch.tensor(floors, dtype=torch.float32)              # 框架慣用 float32
y = torch.tensor(heights, dtype=torch.float32)

reg_model = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1))
loss_fn = nn.MSELoss()
optimizer = optim.SGD(reg_model.parameters(), lr=0.001)    # 注意：lr 太大會發散，這裡用較小步伐

loss_history = []
for epoch in range(200):
    pred = reg_model(X)              # 1. forward
    loss = loss_fn(pred, y)         # 2. loss
    loss.backward()                 # 3. backward：算梯度
    optimizer.step()                # 4. step：更新參數
    optimizer.zero_grad()           # 5. zero_grad：清掉梯度，否則下一輪會累加
    loss_history.append(loss.item())

print("第一個 epoch 損失:", round(loss_history[0], 3))
print("最後一個 epoch 損失:", round(loss_history[-1], 3))

plt.figure(figsize=(6, 4))
plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.title("loss going down means the model is learning")
plt.show()

## 實戰：自造資料的小任務

現在把前面所有積木組裝成一個能跑通的小專案：一個二分類（binary classification）任務。重點不只是看損失數字下降，更要學會用「畫出預測結果」來判斷模型好不好。

幾個實戰概念：
- 訓練（train）與評估（evaluation）的區分：用訓練資料學、再回頭看模型在資料上畫出的決策邊界（decision boundary）。
- 過擬合（overfitting）初步直覺：若模型把雜訊也死記下來、邊界扭曲得太離譜，泛化就會變差。

下面用 numpy 造兩群略有重疊的點，訓練一個分類 MLP，最後把決策邊界畫出來肉眼檢查。

In [ ]:
# 概念：組裝完整二分類專案，訓練後畫出決策邊界並計算正確率
rng = np.random.default_rng(1)

# 自造兩群帶重疊的點：群 0 偏左下、群 1 偏右上
n = 120
g0 = rng.normal(loc=[-1.0, -1.0], scale=1.1, size=(n, 2))
g1 = rng.normal(loc=[1.5, 1.5], scale=1.1, size=(n, 2))
features_np = np.vstack([g0, g1])
labels_np = np.array([0] * n + [1] * n)

X = torch.tensor(features_np, dtype=torch.float32)
y = torch.tensor(labels_np, dtype=torch.long)            # 注意：交叉熵的標籤要用整數類別 long

clf = nn.Sequential(nn.Linear(2, 16), nn.ReLU(), nn.Linear(16, 2))   # 輸出 2 類
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(clf.parameters(), lr=0.1)

for epoch in range(300):
    logits = clf(X)
    loss = loss_fn(logits, y)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

# 評估：取每筆分數最大的類別當預測，與真實標籤比對算正確率
with torch.no_grad():                                    # 技巧：純推論時關掉梯度計算，省記憶體又更快
    pred_class = clf(X).argmax(dim=1)
    accuracy = (pred_class == y).float().mean().item()
print("分類正確率:", round(accuracy, 3))

# 概念：在特徵平面鋪一張網格，對每個格點預測類別，畫出決策邊界
xx, yy = np.meshgrid(np.linspace(-5, 6, 200), np.linspace(-5, 6, 200))
grid = torch.tensor(np.column_stack([xx.ravel(), yy.ravel()]), dtype=torch.float32)
with torch.no_grad():
    zz = clf(grid).argmax(dim=1).numpy().reshape(xx.shape)

plt.figure(figsize=(6, 5))
plt.contourf(xx, yy, zz, alpha=0.25, cmap="coolwarm")    # 背景色塊就是決策邊界劃分的兩區
plt.scatter(g0[:, 0], g0[:, 1], c="tab:blue", s=14, label="class 0")
plt.scatter(g1[:, 0], g1[:, 1], c="tab:red", s=14, label="class 1")
plt.legend(loc="upper left", fontsize=8)
plt.title("decision boundary, acc = %.2f" % accuracy)
plt.show()

## 練習

以下三題由淺入深，皆為複合型但簡短。資料請自己用 numpy 造（仿真即可），完成後對照「驗收」確認輸出。

In [ ]:
# TODO 1 ·（簡單）預測樓層高度（整合：nn.Module 定義 + 訓練迴圈）
#   情境：依「樓層數」預測一棟住宅大樓的「總樓高（公尺）」，兩者大致成正比但帶一點雜訊。
#   要求：
#     1. 用 numpy 自造樓層數與總樓高的資料（加入隨機雜訊），轉成 float32 張量。
#     2. 用 nn.Module 或 nn.Sequential 定義一個小 MLP（迴歸，輸出一個數值），用 MSE 當損失。
#     3. 寫完整五步驟訓練迴圈（forward -> loss -> backward -> step -> zero_grad）跑若干 epoch，記錄損失。
#   驗收：應看到損失隨 epoch 明顯下降，且預測樓高與真實樓高接近成一直線。
# TODO: 在這裡完成


In [ ]:
# TODO 2 ·（中間）建物用途二分類（整合：Sequential + ReLU + 交叉熵 + 訓練迴圈）
#   情境：用兩維特徵（例如基地面積與容積率）判斷一棟建物屬於「住宅」或「商業」，兩類在特徵平面上略有重疊。
#   要求：
#     1. 用 numpy 自造兩群帶重疊的點與對應標籤（標籤用整數類別，轉成 long 張量）。
#     2. 用 nn.Sequential 堆 Linear-ReLU-Linear、輸出兩類，搭配 CrossEntropyLoss 與 optim 優化器訓練。
#     3. 訓練後在特徵平面上畫出決策邊界，並用 argmax 比對真實標籤算出分類正確率。
#   驗收：應看到一條彎曲的決策邊界把兩群大致分開，正確率明顯高於隨機猜測（0.5）。
# TODO: 在這裡完成


In [ ]:
# TODO 3 ·（稍難）容積率非線性迴歸與容量比較（整合：多單元概念 + 一點設計思考）
#   情境：模擬「臨路寬度」與「可建容積率」之間的非線性關係（例如先升後趨緩的曲線），探討網路容量對擬合的影響。
#   要求：
#     1. 用 numpy 自造一條彎曲的非線性關係資料並加雜訊，轉成張量。
#     2. 設計兩個容量不同的 MLP（例如隱藏層較窄 vs. 較寬，或有無 ReLU），各自用相同訓練迴圈訓練。
#     3. 把兩者的擬合曲線與損失曲線疊在一起比較，並用一段文字討論哪個學得較好、是否有過擬合（overfitting）跡象。
#   驗收：應看到含足夠隱藏單元與 ReLU 的模型能貼合彎曲關係，過於簡單的模型只能近似成直線，
#         並能說出容量與擬合的取捨。
# TODO: 在這裡完成


## 小結

- 單層感知器只能畫一條直線；多層感知器靠隱藏層加上非線性激活才有表達力，能逼近彎曲的關係。線性接線性仍是線性，激活才是關鍵。
- nn.Module 把模型拆成 __init__（準備有哪些層）與 forward（資料怎麼流過），這是看懂所有 PyTorch 模型的鑰匙。
- 三種積木分工明確：nn.Linear 做加權組合、ReLU 提供非線性開關、nn.Sequential 把層依序串起來；維度要前後接得起來。
- 損失函式給「錯多少」的分數（迴歸用 MSE、分類用交叉熵），優化器依梯度與學習率往低處走一步。
- 訓練迴圈固定五步驟：forward -> loss -> backward -> optimizer.step -> zero_grad；zero_grad 不能忘，因為梯度預設會累加。
- 評估模型不只看損失數字，更要畫出擬合曲線或決策邊界肉眼檢查，並留意過擬合的跡象。